# Port Mapping & Container Communication

Notebooks 12 and 13 covered how containers connect to each other via Docker networks. This notebook covers the other half: how traffic gets in from the outside world via port mapping, and the full picture of how containers communicate — with the host, with each other, and with external services.

Topics:
- How port mapping works under the hood (iptables DNAT)
- `-p` syntax variants: specific port, random port, interface binding
- `EXPOSE` vs `-p` — what each actually does
- Container-to-container communication (same network, different network)
- Container-to-host communication (`host.docker.internal`)
- Host-to-container communication without port mapping
- Publishing all exposed ports with `-P`
- `docker port` — inspecting active mappings

## 1. How Port Mapping Works

When you publish a port with `-p`, Docker adds an iptables rule that performs **Destination NAT (DNAT)**:

```
External client
    │  TCP to host:8080
    ▼
Host NIC (eth0)  →  iptables DNAT  →  container:80
                      (172.17.0.2:80)
```

The Docker daemon (`dockerd`) manages these rules via `docker-proxy` or directly via iptables — you never write them by hand.

You can inspect the rules yourself:

```bash
# See Docker's NAT rules
sudo iptables -t nat -L DOCKER -n
```

The container does not know it is receiving NATted traffic. From the container's perspective, the connection arrives on its own interface (e.g. `eth0` at `172.17.0.2:80`) as a normal TCP connection.

## 2. Port Mapping Syntax

The `-p` / `--publish` flag has several forms:

| Syntax | Effect |
|--------|--------|
| `-p 8080:80` | Host port 8080 → container port 80 (all interfaces) |
| `-p 80` | Container port 80 → random available host port |
| `-p 127.0.0.1:8080:80` | Host port 8080 on loopback only (not reachable from network) |
| `-p 0.0.0.0:8080:80` | Explicit all-interfaces (same as `-p 8080:80`) |
| `-p 8080:80/udp` | UDP mapping (default is TCP) |
| `-p 8080:80/tcp -p 8080:80/udp` | Both TCP and UDP on the same port |
| `-P` | Map all `EXPOSE`d ports to random host ports |

You can publish multiple ports by repeating `-p`:

```bash
docker run -p 80:80 -p 443:443 nginx:alpine
```

In [ ]:
import time, subprocess

# Specific port mapping
!docker run -d --name nginx-8080 -p 8080:80 nginx:alpine

time.sleep(1)

# Verify it is reachable
!curl -s -o /dev/null -w "HTTP %{http_code} on host:8080\n" http://localhost:8080

# docker port shows the mapping
!docker port nginx-8080

In [ ]:
# Random host port — Docker picks an available ephemeral port
!docker run -d --name nginx-random -p 80 nginx:alpine

time.sleep(1)

# Find which host port was assigned
assigned = subprocess.check_output(
    ["docker", "port", "nginx-random", "80"]
).decode().strip()
print(f"Assigned mapping: {assigned}")

host_port = assigned.split(":")[-1]
!curl -s -o /dev/null -w "HTTP %{{http_code}} on host:{host_port}\n" http://localhost:{host_port}

In [ ]:
# Loopback-only binding — only reachable from the host itself, not the LAN
!docker run -d --name nginx-loopback -p 127.0.0.1:8181:80 nginx:alpine

time.sleep(1)

# Reachable via loopback
!curl -s -o /dev/null -w "Via 127.0.0.1:8181: HTTP %{http_code}\n" http://127.0.0.1:8181

# The docker port output shows the interface
!docker port nginx-loopback

In [ ]:
# Publish ALL exposed ports to random host ports with -P
# nginx:alpine EXPOSEs port 80
!docker run -d --name nginx-P -P nginx:alpine

time.sleep(1)
!docker port nginx-P

!docker rm -f nginx-8080 nginx-random nginx-loopback nginx-P

## 3. `EXPOSE` vs `-p` — Clarifying the Confusion

These two are frequently confused:

| | `EXPOSE` (Dockerfile) | `-p` (docker run) |
|---|---|---|
| **What it does** | Documents which port the app listens on | Actually publishes the port to the host |
| **Effect on traffic** | None — no iptables rules created | Creates DNAT rules — port is reachable |
| **Who reads it** | Operators, `-P` flag, docker inspect | The host kernel (via iptables) |
| **Required?** | No — but good practice | Yes — without it, port is not reachable externally |

```dockerfile
EXPOSE 8000    # tells operators "this app listens on 8000"
               # does NOT make port 8000 reachable
```

```bash
docker run -p 8000:8000 myapp   # NOW port 8000 is reachable
docker run -P myapp             # -P uses EXPOSE to know which ports to map
```

> Think of `EXPOSE` as a README note. `-p` is the actual wire-up.

## 4. Container-to-Container Communication

Containers on the **same user-defined network** communicate directly by container name — no port mapping needed. The traffic stays within the Docker bridge and never reaches the host's public interface.

```bash
# api talks to db on port 5432 — no -p needed for db
docker network create appnet
docker run -d --name db  --network appnet postgres:16
docker run -d --name api --network appnet myapi
# api connects to: postgres://db:5432/mydb
```

The database port (5432) is **never published** to the host — it's only reachable from other containers on `appnet`. This is correct and secure — you don't want your database port exposed to the world.

In [ ]:
!docker network create commnet

# Server: nginx on port 80, NOT published to host
!docker run -d --name srv --network commnet nginx:alpine

# Client: reaches server by name on the internal port — no -p on server
!docker run -d --name cli --network commnet alpine sleep 60

time.sleep(1)

print("Client reaches server by name on port 80 (no host port mapping on server):")
!docker exec cli wget -qO- http://srv:80 2>/dev/null | head -3

# Verify server is NOT reachable from the host (no -p was set)
import socket
try:
    socket.create_connection(("localhost", 80), timeout=1).close()
    print("Reachable from host (unexpected)")
except:
    print("\nServer port 80 is NOT reachable from the host — correct isolation")

!docker rm -f srv cli
!docker network rm commnet

## 5. Container-to-Host Communication

Sometimes a container needs to reach a service running directly on the host (e.g. a database running on the host during development, or a local Kubernetes API server).

### `host.docker.internal`

Docker Desktop (macOS and Windows) automatically provides the DNS name `host.docker.internal` that resolves to the host machine's IP from inside any container.

```bash
# Connect to a postgres instance running on the host
docker run --rm alpine ping -c 2 host.docker.internal
```

On **Linux** with Docker Engine (not Desktop), `host.docker.internal` is not set automatically. You can add it manually:

```bash
docker run --add-host host.docker.internal:host-gateway alpine ping host.docker.internal
```

`host-gateway` is a special Docker keyword that resolves to the host's bridge gateway IP.

In [ ]:
# Check if host.docker.internal resolves (works on Docker Desktop)
result = subprocess.run(
    ["docker", "run", "--rm", "alpine", "nslookup", "host.docker.internal"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("host.docker.internal resolves:")
    print(result.stdout)
else:
    print("host.docker.internal not available — adding manually via --add-host:")
    !docker run --rm \
        --add-host host.docker.internal:host-gateway \
        alpine nslookup host.docker.internal 2>/dev/null | grep -A2 'Name'

## 6. Host-to-Container Without Port Mapping

Even without `-p`, the host can reach a container directly via its **container IP**. This is useful for debugging without opening external ports.

```bash
# Get container IP
docker inspect mycontainer --format '{{.NetworkSettings.IPAddress}}'

# Connect directly (from the host)
curl http://172.17.0.3:80
```

This works because the host has a route to the Docker bridge (`172.17.0.0/16` via `docker0`). Note: on macOS and Windows this **doesn't work** — Docker runs in a VM and the container IPs are inside the VM, not directly reachable from the host.

In [ ]:
# Start a container with NO port mapping
!docker run -d --name no-port-map nginx:alpine

time.sleep(1)

# Get the container's IP
container_ip = subprocess.check_output(
    ["docker", "inspect", "no-port-map",
     "--format", "{{.NetworkSettings.IPAddress}}"]
).decode().strip()
print(f"Container IP: {container_ip}")

# Try to reach it directly from the host (works on Linux, not macOS/Windows)
result = subprocess.run(
    ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
     f"http://{container_ip}:80"],
    capture_output=True, text=True, timeout=3
)
if result.stdout.strip() == "200":
    print(f"Direct host→container connection: HTTP 200 (Linux — bridge is routable)")
else:
    print("Direct host→container not reachable (macOS/Windows — container is in VM)")

!docker rm -f no-port-map

## 7. `docker port` — Inspecting Mappings

In [ ]:
!docker run -d --name port-inspect \
    -p 8080:80 \
    -p 8443:443 \
    nginx:alpine

# Show all port mappings for the container
print("All mappings:")
!docker port port-inspect

# Query a specific container port
print("\nMapping for container port 80:")
!docker port port-inspect 80

!docker rm -f port-inspect

## 8. Communication Matrix

| From → To | Mechanism | Requires |
|-----------|-----------|----------|
| Container → Container (same network) | DNS by name, direct IP | Same user-defined network |
| Container → Container (different network) | Not possible by default | `docker network connect` to share a network |
| External → Container | Port mapping | `-p host_port:container_port` |
| Host → Container (Linux) | Container IP directly | Host has route via docker bridge |
| Host → Container (macOS/Win) | Port mapping or `host.docker.internal` in reverse | `-p` or Docker Desktop |
| Container → Host | `host.docker.internal` or `--add-host host-gateway` | DNS or explicit host entry |
| Container → Internet | Outbound NAT | Default — works unless `--internal` network |

## Summary

- **Port mapping** (`-p host:container`) creates an iptables DNAT rule that forwards traffic arriving at `host:port` to the container's port. Without it, the container's ports are not reachable externally.
- `-p 8080:80` binds all host interfaces. `-p 127.0.0.1:8080:80` binds loopback only — not reachable from the LAN. `-p 80` assigns a random host port.
- `EXPOSE` documents the port — it creates no rules and does not publish anything. `-P` uses `EXPOSE` to know which ports to map to random host ports.
- **Container-to-container** on the same user-defined network: communicate by container name on the internal port — no port mapping needed or wanted.
- **Container → host**: use `host.docker.internal` (Docker Desktop) or `--add-host host.docker.internal:host-gateway` (Linux Docker Engine).
- **Host → container without `-p`**: works on Linux via the container's IP through the docker bridge. Does not work on macOS/Windows where Docker runs in a VM.
- `docker port CONTAINER` lists active port mappings. `docker port CONTAINER 80` queries a specific container port.